In [1]:
import os

In [2]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    movies: Path
    ratings: Path
    tags: Path

In [6]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_processing_config(self) -> DataTransformationConfig:

        config  = self.config.data_processing

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            movies= config.movies,
            tags= config.tags,
            ratings= config.ratings,
            root_dir= config.root_dir
        )

        return data_transformation_config

In [8]:
import pandas as pd

config = ConfigurationManager()
data_processing_config = config.get_data_processing_config()

movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)

[2021-01-01 12:54:09,510: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 12:54:09,520: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 12:54:09,534: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 12:54:09,542: INFO: common: Created directory at: artifacts]
[2021-01-01 12:54:09,546: INFO: common: Created directory at: artifacts/data_transformation]


In [9]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [10]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [11]:
tags_df.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


# Movie Features


## Step 1: One-shot Encode Genre


In [12]:
movies_df['genres'] = movies_df['genres'].str.split('|')
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [13]:
genre_set = set(genre for genres in movies_df['genres'] for genre in genres)

In [14]:
movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9737                 [Action, Animation, Comedy, Fantasy]
9738                         [Animation, Comedy, Fantasy]
9739                                              [Drama]
9740                                  [Action, Animation]
9741                                             [Comedy]
Name: genres, Length: 9742, dtype: object

In [15]:
genre_set

{'(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western'}

In [16]:
for genre in genre_set:
    movies_df[genre] = movies_df['genres'].apply(lambda x: int(genre in x))

In [17]:
movies_df.head()

,movieId,title,genres,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",0,0,1,1,0,0,0,...,0,0,0,1,0,1,0,0,0,1
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",0,0,1,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,5,Father of the Bride Part II (1995),[Comedy],0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [18]:
movies_features = movies_df.drop(['genres', 'title'], axis=1)
print(movies_features.count())
movies_features.head()

movieId               9742
Western               9742
Romance               9742
Fantasy               9742
Adventure             9742
Action                9742
Film-Noir             9742
(no genres listed)    9742
IMAX                  9742
War                   9742
Crime                 9742
Sci-Fi                9742
Mystery               9742
Thriller              9742
Comedy                9742
Horror                9742
Children              9742
Musical               9742
Drama                 9742
Documentary           9742
Animation             9742
dtype: int64


,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,0,0,1,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,1
1,2,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,3,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,4,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


## Step 2: Combine with tags


In [19]:
movie_tags = tags_df.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
movie_tags

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake
...,...,...
1567,183611,Comedy funny Rachel McAdams
1568,184471,adventure Alicia Vikander video game adaptation
1569,187593,Josh Brolin Ryan Reynolds sarcasm
1570,187595,Emilia Clarke star wars


In [20]:
movies_features = movies_features.merge(movie_tags, on='movieId', how='left')
movies_features['tag'] = movies_features['tag'].fillna('')
movies_features

,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation,tag
0,1,0,0,1,1,0,0,0,0,0,...,0,0,1,0,1,0,0,0,1,pixar pixar fun
1,2,0,0,1,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,fantasy magic board game Robin Williams game
2,3,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,moldy old
3,4,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,
4,5,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,pregnancy remake
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,193581,0,0,1,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,1,
9738,193583,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,
9739,193585,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,
9740,193587,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=50)
tag_features = tfidf.fit_transform(movies_features['tag']).toarray()

tag_features_names = [f"tag_{i}" for i in range(tag_features.shape[1])]
tag_feature_df = pd.DataFrame(tag_features, columns=tag_features_names)
tag_feature_df.head()

,tag_0,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,tag_9,...,tag_40,tag_41,tag_42,tag_43,tag_44,tag_45,tag_46,tag_47,tag_48,tag_49
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
movies_features = pd.concat([movies_features.reset_index(drop=True), tag_feature_df], axis=1)

movies_features = movies_features.drop(['tag'], axis=1)
print(movies_features.count())
movies_features.head()

movieId      9742
Western      9742
Romance      9742
Fantasy      9742
Adventure    9742
             ... 
tag_45       9742
tag_46       9742
tag_47       9742
tag_48       9742
tag_49       9742
Length: 71, dtype: int64


,movieId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,tag_40,tag_41,tag_42,tag_43,tag_44,tag_45,tag_46,tag_47,tag_48,tag_49
0,1,0,0,1,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0,0,1,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# User Features


## Step 1: Aggregate Ratings


In [23]:
# Aggregate user ratings
user_ratings = ratings_df.groupby('userId')['rating'].agg(['mean', 'count']).reset_index()
user_ratings.rename(columns={'mean': 'avg_rating', 'count': 'rating_count'}, inplace=True)

user_ratings.head()

,userId,avg_rating,rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


## Step 2: User Preference for Genres


In [24]:
user_genres = ratings_df.merge(movies_df[['movieId'] + list(genre_set)], on='movieId', how='left')
user_genres.head()

,userId,movieId,rating,timestamp,Western,Romance,Fantasy,Adventure,Action,Film-Noir,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,1,4.0,964982703,0,0,1,1,0,0,...,0,0,0,1,0,1,0,0,0,1
1,1,3,4.0,964981247,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,1,6,4.0,964982224,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,1,47,5.0,964983815,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
4,1,50,5.0,964982931,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0


In [25]:
user_genre_prefenrences = user_genres.groupby('userId')[list(genre_set)].mean().reset_index()
user_genre_prefenrences.head()

,userId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,0.000000,0.094828,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,0.137931,0.034483,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,0.000000,0.128205,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,0.004630,0.032407,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,0.068182,0.068182,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


In [26]:
user_features = user_ratings.merge(user_genre_prefenrences, on='userId', how='left')
print(user_features.count())
user_features.head()

userId                610
avg_rating            610
rating_count          610
Western               610
Romance               610
Fantasy               610
Adventure             610
Action                610
Film-Noir             610
(no genres listed)    610
IMAX                  610
War                   610
Crime                 610
Sci-Fi                610
Mystery               610
Thriller              610
Comedy                610
Horror                610
Children              610
Musical               610
Drama                 610
Documentary           610
Animation             610
dtype: int64


,userId,avg_rating,rating_count,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,4.366379,232,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,3.948276,29,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,2.435897,39,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,3.555556,216,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,3.636364,44,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


In [27]:
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [28]:
class DataTransformation:

    def __init__(self, config: DataTransformationConfig) -> None:
        
        self.config = config

    def train_test_spliting(self):

        data = pd.read_csv(self.config.data_path)
        train,test = train_test_split(data)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Data splitted into test and training set")
        logger.info(train.shape)
        logger.info(test.shape)

In [29]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_spliting()

except Exception as e:
    raise e

[2021-01-01 12:55:06,840: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 12:55:06,850: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 12:55:06,867: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 12:55:06,874: INFO: common: Created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_data_transformation_config'